<a href="https://colab.research.google.com/github/1816x/Algoritmos-Aprendizaje-Automatico/blob/main/Practica%20Tema%2010pt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica del martes - Métricas, hiperparámetros y benchmark

**Nombre:** Santiago Rivera Martinez  
**Matrícula:** AL02850849
**Tema 10 - Parte 2**


## Módulo 1. Benchmark multimodelo con métricas cruzadas

Utilicé el dataset Breast Cancer Wisconsin incluido en Scikit-learn. Comparé Regresión Logística y SVM con kernel RBF usando la misma validación cruzada estratificada de cinco particiones. Evalué accuracy, F1-score y ROC-AUC para no depender de una sola métrica.


In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# load the classification dataset
data = load_breast_cancer()
x = data.data
y = data.target

# define candidate pipelines under the same preprocessing protocol
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=5000, random_state=42)),
    ]),
    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf", probability=True, random_state=42)),
    ]),
}

# evaluate each model with the same stratified folds and metrics
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
}
summary = []

for name, model in models.items():
    results = cross_validate(
        model,
        x,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )
    summary.append({
        "Model": name,
        "Accuracy mean": results["test_accuracy"].mean(),
        "F1 mean": results["test_f1"].mean(),
        "ROC-AUC mean": results["test_roc_auc"].mean(),
        "F1 std": results["test_f1"].std(),
    })

benchmark_df = pd.DataFrame(summary).sort_values("F1 mean", ascending=False)
print(benchmark_df.round(4).to_string(index=False))


              Model  Accuracy mean  F1 mean  ROC-AUC mean  F1 std
            SVM RBF         0.9772   0.9820        0.9945  0.0128
Logistic Regression         0.9737   0.9794        0.9953  0.0127


Ordené el benchmark por F1-score medio. Además de comparar el promedio, incluí la desviación estándar de F1 para revisar la estabilidad entre particiones. Mantener el mismo StratifiedKFold, las mismas métricas y el mismo dataset hace que la comparación sea justa.


## Módulo 2. GridSearchCV y evaluación en test

Separé 20% de los datos como conjunto de prueba y conservé la proporción de clases mediante estratificación. La búsqueda de hiperparámetros se realiza solo sobre entrenamiento con validación cruzada de cinco particiones. El pipeline integra escalamiento y SVM, evitando que el escalador se ajuste con datos de validación o prueba.


In [2]:
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split

# create a fixed train-test split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

# define the pipeline and a bounded hyperparameter grid
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(kernel="rbf", probability=True, random_state=42)),
])

parameter_grid = {
    "classifier__C": [0.1, 1, 10],
    "classifier__gamma": [0.001, 0.01, 0.1],
}

# tune only on training data using five-fold cross-validation
grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=parameter_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    return_train_score=True,
)
grid.fit(x_train, y_train)

# evaluate the selected estimator once on the reserved test set
y_pred = grid.predict(x_test)

print("Best parameters:", grid.best_params_)
print(f"Best mean CV F1-score: {grid.best_score_:.4f}")
print("\nClassification report on test data:")
print(classification_report(y_test, y_pred, target_names=data.target_names))


Best parameters: {'classifier__C': 10, 'classifier__gamma': 0.01}
Best mean CV F1-score: 0.9844

Classification report on test data:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



El conjunto de prueba se reservó para la evaluación final: no se utilizó para elegir C ni gamma. GridSearchCV probó las nueve combinaciones de la malla y seleccionó la que obtuvo el mayor F1-score promedio en validación cruzada.


## Módulo 3. Diccionario de evidencia para reporte técnico

Concentré la configuración, el protocolo de validación y las métricas del modelo seleccionado en una estructura reproducible. Esta bitácora permite revisar qué se ejecutó, con qué semilla y qué resultado obtuvo el modelo en validación y prueba.


In [3]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# collect the selected model and its reproducible evidence
best_model = grid.best_estimator_
test_f1 = f1_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_accuracy = accuracy_score(y_test, y_pred)

technical_report = {
    "experiment": "Benchmark cierre Tema 10",
    "dataset": "Breast Cancer Wisconsin",
    "dataset_records": int(x.shape[0]),
    "features": int(x.shape[1]),
    "target": "diagnosis (0 = malignant, 1 = benign)",
    "train_test_split": "80/20 stratified, random_state=42",
    "validation": "StratifiedKFold, 5 folds",
    "selection_metric": "F1-score",
    "best_model": type(best_model.named_steps["classifier"]).__name__,
    "best_hyperparameters": grid.best_params_,
    "f1_cv_mean": round(grid.best_score_, 4),
    "f1_test": round(test_f1, 4),
    "precision_test": round(test_precision, 4),
    "recall_test": round(test_recall, 4),
    "accuracy_test": round(test_accuracy, 4),
}

for key, value in technical_report.items():
    print(f"{key}: {value}")


experiment: Benchmark cierre Tema 10
dataset: Breast Cancer Wisconsin
dataset_records: 569
features: 30
target: diagnosis (0 = malignant, 1 = benign)
train_test_split: 80/20 stratified, random_state=42
validation: StratifiedKFold, 5 folds
selection_metric: F1-score
best_model: SVC
best_hyperparameters: {'classifier__C': 10, 'classifier__gamma': 0.01}
f1_cv_mean: 0.9844
f1_test: 0.9861
precision_test: 0.9861
recall_test: 0.9861
accuracy_test: 0.9825


El diccionario final funciona como una evidencia técnica del experimento. Registra el modelo, hiperparámetros, partición, estrategia de validación y métricas. Con esta información se puede repetir el procedimiento y explicar por qué se seleccionó la configuración final.


## Resumen Adicional

Este notebook detalló un protocolo de evaluación de modelos ML, usando el dataset de cáncer de mama. Se compararon Regresión Logística y SVM RBF, mostrando este último una performance superior. La optimización de hiperparámetros se realizó con GridSearchCV sobre los datos de entrenamiento.

Una evaluación final en un conjunto de prueba reservado confirmó la capacidad de generalización. Finalmente, se creó un reporte técnico para documentar y reproducir los resultados del modelo óptimo.